In [1]:
import sys
sys.path.insert(0, '../..')

import os
import json
import time
import warnings
warnings.filterwarnings('ignore')

import boto3
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

load_dotenv('../../.env')

from src.utils.config import settings
from src.utils.logger import get_logger

log = get_logger("s3_pipeline")

PROC  = '../../data/processed/'
FEAT  = '../../data/features/'
RAW   = '../../data/raw/'
DELTA = '../../data/delta_lake/'

print("✅ Imports ready")

✅ Imports ready


In [2]:
# Connect to s3

# Create S3 client
s3 = boto3.client(
    's3',
    aws_access_key_id     = settings.AWS_ACCESS_KEY_ID,
    aws_secret_access_key = settings.AWS_SECRET_ACCESS_KEY,
    region_name           = settings.AWS_REGION
)

BUCKET = settings.S3_BUCKET

# Test connection
try:
    response = s3.list_buckets()
    buckets  = [b['Name'] for b in response['Buckets']]
    print(f"✅ Connected to AWS S3")
    print(f"   Available buckets: {buckets}")
    print(f"   Using bucket     : {BUCKET}")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("   Check your .env AWS credentials")

✅ Connected to AWS S3
   Available buckets: ['demo-cors-d', 'demobucket-adarsha', 'production-recsys-data']
   Using bucket     : production-recsys-data


In [4]:
## Define S3 Data Lake Structure

# S3 folder structure — mirrors local structure
S3_STRUCTURE = {
    "raw":       f"data-lake/raw/",
    "processed": f"data-lake/processed/",
    "features":  f"data-lake/features/",
    "delta":     f"data-lake/delta/",
    "models":    f"data-lake/models/",
}

print("S3 Data Lake Structure:")
print(f"  s3://{BUCKET}/")
for layer, path in S3_STRUCTURE.items():
    print(f"    {path:<35} ← {layer} layer")

print("""
LAYER RESPONSIBILITIES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
raw/         Original CSV files — never modified
processed/   Cleaned CSVs from Day 3
features/    Engineered features from Day 4
delta/       Delta Lake tables from Day 6
models/      Trained model artifacts (Week 3+)
""")

S3 Data Lake Structure:
  s3://production-recsys-data/
    data-lake/raw/                      ← raw layer
    data-lake/processed/                ← processed layer
    data-lake/features/                 ← features layer
    data-lake/delta/                    ← delta layer
    data-lake/models/                   ← models layer

LAYER RESPONSIBILITIES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
raw/         Original CSV files — never modified
processed/   Cleaned CSVs from Day 3
features/    Engineered features from Day 4
delta/       Delta Lake tables from Day 6
models/      Trained model artifacts (Week 3+)



In [5]:
## Helper function

def upload_file(local_path: str,
                s3_key: str,
                description: str = "") -> bool:
    """Upload single file to S3 with progress logging"""
    try:
        file_size = os.path.getsize(local_path) / 1024 / 1024
        log.info(f"Uploading {description} "
                 f"({file_size:.1f} MB) → s3://{BUCKET}/{s3_key}")

        start = time.time()
        s3.upload_file(local_path, BUCKET, s3_key)
        elapsed = time.time() - start

        log.info(f"✅ Uploaded in {elapsed:.1f}s")
        return True
    except Exception as e:
        log.error(f"❌ Failed to upload {local_path}: {e}")
        return False


def upload_folder(local_folder: str,
                  s3_prefix: str,
                  extensions: list = None) -> dict:
    """Upload all files in a folder to S3"""
    results  = {"success": [], "failed": []}
    folder   = Path(local_folder)

    for file_path in sorted(folder.iterdir()):
        if not file_path.is_file():
            continue
        if extensions and file_path.suffix not in extensions:
            continue

        s3_key = f"{s3_prefix}{file_path.name}"
        ok = upload_file(
            str(file_path), s3_key, file_path.name)

        if ok:
            results["success"].append(file_path.name)
        else:
            results["failed"].append(file_path.name)

    return results


def list_s3_folder(prefix: str) -> list:
    """List all files in an S3 prefix"""
    response = s3.list_objects_v2(
        Bucket=BUCKET, Prefix=prefix)
    objects = response.get('Contents', [])
    return [
        {
            "key":      obj['Key'],
            "size_mb":  round(obj['Size'] / 1024 / 1024, 2),
            "modified": obj['LastModified'].strftime(
                            "%Y-%m-%d %H:%M")
        }
        for obj in objects
    ]


print("✅ Upload helpers defined")

✅ Upload helpers defined


In [6]:
## Upload Raw Data

print("Uploading raw data to S3...")
print("─" * 50)

raw_files = [
    ('movies_metadata.csv', 'movie metadata'),
    ('credits.csv',         'cast and crew'),
    ('keywords.csv',        'plot keywords'),
    ('links.csv',           'ID mappings'),
    ('links_small.csv',     'small ID mappings'),
    ('ratings_small.csv',   'sample ratings'),
    ('ratings.csv',         'full ratings 26M'),   
]

raw_results = {"success": [], "failed": []}

for filename, description in raw_files:
    local_path = RAW + filename
    s3_key     = S3_STRUCTURE["raw"] + filename

    if not os.path.exists(local_path):
        print(f"⚠️  Skipping {filename} — not found locally")
        continue

    ok = upload_file(local_path, s3_key, description)
    if ok:
        raw_results["success"].append(filename)
    else:
        raw_results["failed"].append(filename)

print(f"\n✅ Raw upload complete")
print(f"   Uploaded : {len(raw_results['success'])} files")
print(f"   Failed   : {len(raw_results['failed'])} files")

Uploading raw data to S3...
──────────────────────────────────────────────────
2026-06-02 17:32:07 | INFO | s3_pipeline | Uploading movie metadata (32.8 MB) → s3://production-recsys-data/data-lake/raw/movies_metadata.csv
2026-06-02 17:32:12 | INFO | s3_pipeline | ✅ Uploaded in 5.2s
2026-06-02 17:32:12 | INFO | s3_pipeline | Uploading cast and crew (181.1 MB) → s3://production-recsys-data/data-lake/raw/credits.csv
2026-06-02 17:32:28 | INFO | s3_pipeline | ✅ Uploaded in 16.4s
2026-06-02 17:32:28 | INFO | s3_pipeline | Uploading plot keywords (5.9 MB) → s3://production-recsys-data/data-lake/raw/keywords.csv
2026-06-02 17:32:31 | INFO | s3_pipeline | ✅ Uploaded in 2.4s
2026-06-02 17:32:31 | INFO | s3_pipeline | Uploading ID mappings (0.9 MB) → s3://production-recsys-data/data-lake/raw/links.csv
2026-06-02 17:32:31 | INFO | s3_pipeline | ✅ Uploaded in 0.3s
2026-06-02 17:32:31 | INFO | s3_pipeline | Uploading small ID mappings (0.2 MB) → s3://production-recsys-data/data-lake/raw/links_small

In [7]:
## Upload Processed Data

print("Uploading processed data to S3...")
print("─" * 50)

processed_results = upload_folder(
    local_folder = PROC,
    s3_prefix    = S3_STRUCTURE["processed"],
    extensions   = ['.csv', '.json']
)

print(f"\n✅ Processed upload complete")
print(f"   Uploaded : {len(processed_results['success'])} files")
if processed_results['failed']:
    print(f"   Failed   : {processed_results['failed']}")

Uploading processed data to S3...
──────────────────────────────────────────────────
2026-06-02 17:33:32 | INFO | s3_pipeline | Uploading cleaning_report.json (0.0 MB) → s3://production-recsys-data/data-lake/processed/cleaning_report.json
2026-06-02 17:33:33 | INFO | s3_pipeline | ✅ Uploaded in 0.1s
2026-06-02 17:33:33 | INFO | s3_pipeline | Uploading credits_cleaned.csv (183.8 MB) → s3://production-recsys-data/data-lake/processed/credits_cleaned.csv
2026-06-02 17:33:49 | INFO | s3_pipeline | ✅ Uploaded in 16.4s
2026-06-02 17:33:49 | INFO | s3_pipeline | Uploading eda_summary.json (0.0 MB) → s3://production-recsys-data/data-lake/processed/eda_summary.json
2026-06-02 17:33:49 | INFO | s3_pipeline | ✅ Uploaded in 0.1s
2026-06-02 17:33:49 | INFO | s3_pipeline | Uploading keywords_cleaned.csv (7.8 MB) → s3://production-recsys-data/data-lake/processed/keywords_cleaned.csv
2026-06-02 17:33:51 | INFO | s3_pipeline | ✅ Uploaded in 1.8s
2026-06-02 17:33:51 | INFO | s3_pipeline | Uploading links

In [8]:
## Upload Feature Files

print("Uploading feature files to S3...")
print("─" * 50)

feature_results = upload_folder(
    local_folder = FEAT,
    s3_prefix    = S3_STRUCTURE["features"],
    extensions   = ['.csv', '.json',
                    '.joblib', '.npz']
)

print(f"\n✅ Feature upload complete")
print(f"   Uploaded : {len(feature_results['success'])} files")
if feature_results['failed']:
    print(f"   Failed   : {feature_results['failed']}")

Uploading feature files to S3...
──────────────────────────────────────────────────
2026-06-02 17:34:08 | INFO | s3_pipeline | Uploading genre_encoder.joblib (0.0 MB) → s3://production-recsys-data/data-lake/features/genre_encoder.joblib
2026-06-02 17:34:08 | INFO | s3_pipeline | ✅ Uploaded in 0.3s
2026-06-02 17:34:08 | INFO | s3_pipeline | Uploading genre_features.csv (2.0 MB) → s3://production-recsys-data/data-lake/features/genre_features.csv
2026-06-02 17:34:09 | INFO | s3_pipeline | ✅ Uploaded in 0.5s
2026-06-02 17:34:09 | INFO | s3_pipeline | Uploading id_mappings.joblib (0.3 MB) → s3://production-recsys-data/data-lake/features/id_mappings.joblib
2026-06-02 17:34:09 | INFO | s3_pipeline | ✅ Uploaded in 0.1s
2026-06-02 17:34:09 | INFO | s3_pipeline | Uploading interaction_matrix.npz (0.2 MB) → s3://production-recsys-data/data-lake/features/interaction_matrix.npz
2026-06-02 17:34:09 | INFO | s3_pipeline | ✅ Uploaded in 0.2s
2026-06-02 17:34:09 | INFO | s3_pipeline | Uploading movie_f

In [9]:
# Verify S3 Contents

print("S3 DATA LAKE CONTENTS")
print("=" * 60)

total_size = 0

for layer, prefix in S3_STRUCTURE.items():
    files = list_s3_folder(prefix)
    if not files:
        continue

    layer_size = sum(f['size_mb'] for f in files)
    total_size += layer_size

    print(f"\n📁 {prefix}")
    print(f"   Files: {len(files)} | "
          f"Size: {layer_size:.1f} MB")
    for f in files:
        print(f"   {f['key'].split('/')[-1]:<45} "
              f"{f['size_mb']:>8.1f} MB")

print(f"\n{'='*60}")
print(f"Total S3 data lake size: {total_size:.1f} MB")

S3 DATA LAKE CONTENTS

📁 data-lake/raw/
   Files: 7 | Size: 900.0 MB
   credits.csv                                      181.1 MB
   keywords.csv                                       5.9 MB
   links.csv                                          0.9 MB
   links_small.csv                                    0.2 MB
   movies_metadata.csv                               32.9 MB
   ratings.csv                                      676.7 MB
   ratings_small.csv                                  2.3 MB

📁 data-lake/processed/
   Files: 11 | Size: 248.1 MB
   cleaning_report.json                               0.0 MB
   credits_cleaned.csv                              183.8 MB
   eda_summary.json                                   0.0 MB
   keywords_cleaned.csv                               7.8 MB
   links_cleaned.csv                                  0.8 MB
   links_small_cleaned.csv                            0.2 MB
   movie_stats_full.csv                               3.3 MB
   movies_master.csv   

In [10]:
# Read Data Back From S3

import io

print("Reading data back from S3...\n")

# Read processed ratings from S3
print("1. Reading ratings_cleaned.csv from S3...")
start = time.time()

obj = s3.get_object(
    Bucket = BUCKET,
    Key    = S3_STRUCTURE["processed"] + "ratings_cleaned.csv"
)
ratings_from_s3 = pd.read_csv(
    io.BytesIO(obj['Body'].read()))

elapsed = time.time() - start
print(f"   ✅ Loaded {len(ratings_from_s3):,} rows "
      f"in {elapsed:.2f}s")
print(f"   Shape: {ratings_from_s3.shape}")
print(f"   Columns: {list(ratings_from_s3.columns)}")

# Read movies master from S3
print("\n2. Reading movies_master.csv from S3...")
start = time.time()

obj = s3.get_object(
    Bucket = BUCKET,
    Key    = S3_STRUCTURE["processed"] + "movies_master.csv"
)
movies_from_s3 = pd.read_csv(
    io.BytesIO(obj['Body'].read()),
    low_memory=False
)

elapsed = time.time() - start
print(f"   ✅ Loaded {len(movies_from_s3):,} rows "
      f"in {elapsed:.2f}s")
print(f"   Shape: {movies_from_s3.shape}")

print("\n✅ S3 round-trip verified — data flows correctly")

Reading data back from S3...

1. Reading ratings_cleaned.csv from S3...
   ✅ Loaded 100,004 rows in 0.66s
   Shape: (100004, 4)
   Columns: ['userId', 'movieId', 'rating', 'timestamp']

2. Reading movies_master.csv from S3...
   ✅ Loaded 45,454 rows in 5.52s
   Shape: (45454, 24)

✅ S3 round-trip verified — data flows correctly


In [11]:
# Connect DVC to S3 Remote

import subprocess

def run_cmd(cmd: str) -> str:
    result = subprocess.run(
        cmd, shell=True,
        capture_output=True, text=True
    )
    return result.stdout + result.stderr

# Configure DVC S3 remote
bucket = settings.S3_BUCKET
region = settings.AWS_REGION

commands = [
    f"cd ../.. && dvc remote add -f -d s3remote "
    f"s3://{bucket}/dvc-store",

    f"cd ../.. && dvc remote modify s3remote "
    f"region {region}",

    "cd ../.. && dvc remote list",
]

for cmd in commands:
    output = run_cmd(cmd)
    print(f"$ {cmd.split('&&')[-1].strip()}")
    print(f"  {output.strip()}")

print("✅ DVC S3 remote configured")

$ dvc remote add -f -d s3remote s3://production-recsys-data/dvc-store
  Setting 's3remote' as a default remote.
$ dvc remote modify s3remote region us-east-1
  
$ dvc remote list
  myremote	s3://production-recsys-data/dvc-store
s3remote	s3://production-recsys-data/dvc-store
✅ DVC S3 remote configured


In [12]:
# Push Data to DVC Remote

print("Pushing data to DVC S3 remote...")
print("This uploads all tracked data to S3...\n")

output = run_cmd("cd ../.. && dvc push")
print(output)

print("""
✅ DVC push complete

Your data is now versioned in S3.
Anyone can reproduce your exact dataset with:
  git clone <repo>
  dvc pull
""")

Pushing data to DVC S3 remote...
This uploads all tracked data to S3...

23 files pushed


✅ DVC push complete

Your data is now versioned in S3.
Anyone can reproduce your exact dataset with:
  git clone <repo>
  dvc pull



In [13]:
# Create Data Lake manifest
# Create a manifest of everything in the data lake
manifest = {
    "project":    "production-recsys",
    "created":    pd.Timestamp.now().isoformat(),
    "bucket":     BUCKET,
    "region":     settings.AWS_REGION,
    "layers": {}
}

for layer, prefix in S3_STRUCTURE.items():
    files = list_s3_folder(prefix)
    manifest["layers"][layer] = {
        "prefix":     f"s3://{BUCKET}/{prefix}",
        "file_count": len(files),
        "total_mb":   round(
            sum(f['size_mb'] for f in files), 2),
        "files": [f['key'].split('/')[-1]
                  for f in files]
    }

# Save manifest locally and to S3
manifest_path = PROC + 'data_lake_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

# Upload manifest to S3
s3.upload_file(
    manifest_path, BUCKET,
    'data-lake/manifest.json'
)

print("✅ Data lake manifest saved")
print(json.dumps(manifest, indent=2))


✅ Data lake manifest saved
{
  "project": "production-recsys",
  "created": "2026-06-02T17:39:41.870691",
  "bucket": "production-recsys-data",
  "region": "us-east-1",
  "layers": {
    "raw": {
      "prefix": "s3://production-recsys-data/data-lake/raw/",
      "file_count": 7,
      "total_mb": 900.03,
      "files": [
        "credits.csv",
        "keywords.csv",
        "links.csv",
        "links_small.csv",
        "movies_metadata.csv",
        "ratings.csv",
        "ratings_small.csv"
      ]
    },
    "processed": {
      "prefix": "s3://production-recsys-data/data-lake/processed/",
      "file_count": 11,
      "total_mb": 248.12,
      "files": [
        "cleaning_report.json",
        "credits_cleaned.csv",
        "eda_summary.json",
        "keywords_cleaned.csv",
        "links_cleaned.csv",
        "links_small_cleaned.csv",
        "movie_stats_full.csv",
        "movies_master.csv",
        "ratings_cleaned.csv",
        "user_stats_full.csv",
        "warehouse_s

In [14]:
print("""
╔══════════════════════════════════════════════════════════════╗
║                   WEEK 1 COMPLETE ✅                        ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  Day 1  Environment + GitHub + Docker setup                 ║
║  Day 2  EDA — sparsity, bias, distributions                 ║
║  Day 3  Data cleaning — master movie table                  ║
║  Day 4  Feature engineering — TF-IDF, genres, users        ║
║  Day 5  PostgreSQL star schema warehouse                    ║
║  Day 6  PySpark ETL — 26M ratings, Delta Lake              ║
║  Day 7  S3 data lake — upload, verify, DVC remote          ║
║                                                              ║
╠══════════════════════════════════════════════════════════════╣
║  DATA PLATFORM STACK                                         ║
║                                                              ║
║  Storage    Delta Lake + S3 + PostgreSQL                    ║
║  Processing PySpark (26M rows) + Pandas (dev)              ║
║  Features   TF-IDF + genre + numerical + temporal          ║
║  Versioning DVC → S3 remote                                ║
║  Quality    Great Expectations (Day 1 config)              ║
║  Tracking   MLflow experiment server running               ║
║  Warehouse  Star schema — 6 tables loaded                  ║
║                                                              ║
╠══════════════════════════════════════════════════════════════╣
║  NEXT: Week 2 — Recommendation Models                       ║
║                                                              ║
║  Day 8   CF baselines from scratch                          ║
║  Day 9   SVD + ALS                                          ║
║  Day 10  GRank generative retrieval                        ║
║  Day 11  Qdrant + CLIP multimodal                          ║
║  Day 12  LLM query expansion + RAG                         ║
║  Day 13  Baseline benchmarking                              ║
║  Day 14  Week 2 review + cleanup                           ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════╗
║                   WEEK 1 COMPLETE ✅                        ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  Day 1  Environment + GitHub + Docker setup                 ║
║  Day 2  EDA — sparsity, bias, distributions                 ║
║  Day 3  Data cleaning — master movie table                  ║
║  Day 4  Feature engineering — TF-IDF, genres, users        ║
║  Day 5  PostgreSQL star schema warehouse                    ║
║  Day 6  PySpark ETL — 26M ratings, Delta Lake              ║
║  Day 7  S3 data lake — upload, verify, DVC remote          ║
║                                                              ║
╠══════════════════════════════════════════════════════════════╣
║  DATA PLATFORM STACK                                         ║
║                                                              ║
║  Storage    Delta Lake + S3 + Post